# VisionTrack — reproduce the synthetic study in your browser

[VisionTrack](https://github.com/hulagerushikesh/visiontrack) is a from-scratch
multi-object tracker (8-state Kalman + O(n³) Hungarian + ByteTrack, NumPy-only
core) used as a **controlled, significance-tested study** of when the field's
standard tricks actually help online tracking.

This notebook reproduces the **synthetic** half of the study end-to-end — **no
dataset download, CPU-only, a couple of minutes**. It runs the exact same
commands as `make reproduce-synth` and shows:

1. a **sanity check** — baseline vs. an identical copy gives Δ=0, p=1.00 (the
   harness isn't fooling itself), and
2. the harness **discriminating real ablations** under seed variance + paired
   Wilcoxon significance (removing two-stage recovery / the Mahalanobis gate).

Everything is seeded and pinned by a config hash, so your numbers should match.

🔗 Live site: [visiontrack.hulage.in](https://visiontrack.hulage.in) ·
the write-up: [/writeup](https://visiontrack.hulage.in/writeup) ·
benchmark: [/benchmark](https://visiontrack.hulage.in/benchmark)

## 1 · Setup — clone + install (the `experiments` extra, ~30s)

In [ ]:
import os
if not os.path.isdir('visiontrack'):
    !git clone --depth 1 https://github.com/hulagerushikesh/visiontrack.git
os.chdir('/content/visiontrack' if os.path.isdir('/content/visiontrack') else 'visiontrack')
print('cwd:', os.getcwd())
# The NumPy core needs nothing extra; the harness (pandas/pyarrow/scipy/pyyaml/
# matplotlib) lives in the lazily-imported [experiments] extra.
!pip -q install -e '.[experiments]'

## 2 · Run the synthetic sweep + significance analysis

`run_matrix` sweeps each tracker variant over seeds into a tidy `parquet`;
`analyze` compares every variant to the baseline **paired** (same scenes/seeds)
with a Wilcoxon test.

In [ ]:
!python -m experiments.run_matrix --config experiments/configs/synth_baseline.yaml --out results_synth.parquet
!python -m experiments.analyze --results results_synth.parquet --out-md docs/results_synth.md --out-fig reproduce_synth.png

## 3 · The results — table + figure

In [ ]:
from IPython.display import Markdown, Image, display
display(Markdown(open('docs/results_synth.md').read()))
display(Image('reproduce_synth.png'))

**What you're seeing.** `baseline` vs `baseline_copy` gives Δ=0 at p=1.00 — the
paired test is honest. `no_recovery` and `no_gating` are *real* ablations and the
harness flags them as significant (p<0.05), at small effect sizes — exactly the
resolution the study relies on to tell a genuine effect from noise.

## 4 · RQ3 probe — uncertainty-aware association (synthetic)

The same harness on the RQ3 configuration, compared to the `motion_gate`
baseline. On the real study this is where calibrating the filter turns out to be
*harmful* under detector noise — one of the honest negatives.

In [ ]:
!python -m experiments.run_matrix --config experiments/configs/rq3_uncertainty_synth.yaml --out results_rq3_synth.parquet
!python -m experiments.analyze --results results_rq3_synth.parquet --baseline motion_gate

## Where to go next

- **The whole story** (why appearance refuses to hurt, why a better motion
  predictor made a worse tracker): [the write-up](https://visiontrack.hulage.in/writeup).
- **The honest MOT benchmark** (leaderboard + paired significance + error
  taxonomy) on [synthetic](https://visiontrack.hulage.in/benchmark) or
  [real DanceTrack](https://visiontrack.hulage.in/benchmark/dancetrack) data.
- **The code + mini-paper README:**
  [github.com/hulagerushikesh/visiontrack](https://github.com/hulagerushikesh/visiontrack).
- Run the full **real-data** reproduction locally after a one-time cache build —
  see `docs/PHASE0.md`.